# Multiverse Hybrid v3.0 — Stage 3 Ticket Filter 2000 v2

v1のDrive連続書き込みを廃止した耐障害版です。科学ルールは変更していません。

- 557MB Stage 2 catalogはDriveから一度だけローカルへコピー
- 250 records × 16 resumable chunks
- 中断しても完成済みchunkから再開
- Driveへの細かい連続書き込みを廃止
- 巨大ZIP / 自動ダウンロードなし
- runtime系失敗は最大3回自動再試行
- RESULT / PAYOUT / Settlement / realized ROIなし

iPhoneでは **ランタイム → すべてのセルを実行** だけで構いません。


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, shutil, hashlib, json, time, os

drive.mount('/content/drive')
MY=Path('/content/drive/MyDrive')
REPO=Path('/content/multiverse-research-stage3-v2')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.check_call(['git','clone','--depth','1','https://github.com/fufufu1116/multiverse-research.git',str(REPO)])

EXPECTED={
 'v3/historical_all_market/stage3_ticket_filter_diagnostics_v2.py':'91922c3f2f3da4f2af00e8e6ffcba2c6eaf041df',
 'v3/historical_all_market/governance/STAGE3_TICKET_FILTER_FAMILY_PREREG_v1.md':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
 'v3/historical_all_market/governance/COLAB_IPHONE_RUNTIME_RELIABILITY_STANDARD_v1.md':'73db0eff65c9fa8ee4e97420ff3069481d359d2e',
 'v3/historical_all_market/runtime_receipts/ALL_MARKET_STAGE3_RESUMABLE_V2_STATIC_SELF_CHECK_v1.json':'b1395e668abfddf4d773f541c2a4f1556a35ba3d',
}
for rel,exp in EXPECTED.items():
    obs=subprocess.check_output(['git','-C',str(REPO),'hash-object',rel],text=True).strip()
    if obs!=exp: raise RuntimeError(f'FAIL-CLOSED Git blob mismatch {rel}: {obs} != {exp}')
print('✅ STAGE3 v2 EXACT BINDINGS PASS')

S2=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'/'DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl'
S2R=MY/'MULTIVERSE_ALL_MARKET_STAGE2_PRICE_EV_v1'/'STAGE2_PRICE_EV_RECEIPT_v1.json'
OUT=MY/'MULTIVERSE_ALL_MARKET_STAGE3_TICKET_FILTER_v2'
CHUNKS=OUT/'CHUNKS'
SUMMARY=OUT/'STAGE3_TICKET_FILTER_SUMMARY_v2.csv'
QUALITY=OUT/'STAGE3_TICKET_FILTER_DIAGNOSTICS_QUALITY_v2.json'
RECEIPT=OUT/'STAGE3_TICKET_FILTER_RECEIPT_v2.json'
FATAL=OUT/'STAGE3_RUNTIME_FATAL_v2.json'
LOG=OUT/'STAGE3_TICKET_FILTER_RUN_LOG_v2.txt'
OUT.mkdir(parents=True,exist_ok=True); CHUNKS.mkdir(parents=True,exist_ok=True)
EXPECTED_S2_SHA='34ad32bed6e8b4d700864c46f4533bef1da254c7d87dc7ffe6ec266fd74530dc'
EXPECTED_S2_SIZE=557500538

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for c in iter(lambda:f.read(1<<20),b''): h.update(c)
    return h.hexdigest()

# Existing PASS fast-path: no 557MB copy/rescan.
existing_ok=False
if RECEIPT.is_file() and QUALITY.is_file() and SUMMARY.is_file():
    try:
        r=json.loads(RECEIPT.read_text(encoding='utf-8')); q=json.loads(QUALITY.read_text(encoding='utf-8'))
        existing_ok=(
          r.get('status')=='PASS' and q.get('status')=='PASS' and
          r.get('summary_csv_sha256')==sha256(SUMMARY) and
          r.get('quality_sha256')==sha256(QUALITY) and
          q.get('completed_chunks')==16 and q.get('input_rows')==4000 and q.get('unique_races')==2000 and
          q.get('profile_selection_performed') is False and q.get('market_specific_threshold_tuning_performed') is False and
          q.get('result_access') is False and q.get('payout_access') is False and q.get('settlement_access') is False and
          q.get('realized_roi_computed') is False and q.get('scientific_trial_count')==0 and
          q.get('ECON_HOLDOUT1000')=='SEALED'
        )
    except Exception:
        existing_ok=False

if existing_ok:
    print('✅ STAGE3 v2 ALREADY PASS — 再計算不要')
else:
    if not S2.is_file() or not S2R.is_file():
        raise RuntimeError(f'FAIL-CLOSED Stage2 input missing catalog={S2.exists()} receipt={S2R.exists()}')
    s2r=json.loads(S2R.read_text(encoding='utf-8'))
    if s2r.get('status')!='PASS' or s2r.get('catalog_sha256')!=EXPECTED_S2_SHA:
        raise RuntimeError('FAIL-CLOSED Stage2 receipt binding')
    if s2r.get('result_access') is not False or s2r.get('settlement_access') is not False or s2r.get('realized_roi_computed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage2 firewall')

    LOCAL=Path('/content/DEV2000_ALL_MARKET_PRICE_EV_CATALOG_v1.jsonl')
    # Copy heavy input once to local disk; retry transient Drive reads automatically.
    local_ready=LOCAL.is_file() and LOCAL.stat().st_size==EXPECTED_S2_SIZE
    if not local_ready:
        if LOCAL.exists(): LOCAL.unlink()
        last=None
        for attempt in range(1,4):
            try:
                print(f'▶ Stage2 catalog local copy attempt {attempt}/3')
                shutil.copyfile(S2,LOCAL)
                if LOCAL.stat().st_size!=EXPECTED_S2_SIZE:
                    raise RuntimeError(f'local copy size mismatch {LOCAL.stat().st_size} != {EXPECTED_S2_SIZE}')
                local_ready=True; break
            except Exception as e:
                last=e
                if LOCAL.exists(): LOCAL.unlink()
                time.sleep(5*attempt)
        if not local_ready: raise RuntimeError(f'RUNTIME_RETRYABLE Stage2 local copy failed after 3 attempts: {last}')
    print('✅ Stage2 catalog local copy ready')

    ENGINE=REPO/'v3/historical_all_market/stage3_ticket_filter_diagnostics_v2.py'
    cmd=['python',str(ENGINE),str(LOCAL),str(CHUNKS),str(SUMMARY),str(QUALITY),str(FATAL)]
    logs=[]; success=False
    for attempt in range(1,4):
        print(f'▶ STAGE3 v2 resumable run attempt {attempt}/3')
        proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
        logs.append(f'=== attempt {attempt} return={proc.returncode} ===\n'+proc.stdout)
        print('\n'.join(proc.stdout.splitlines()[-50:]))
        LOG.write_text('\n\n'.join(logs),encoding='utf-8')
        if proc.returncode==0:
            success=True; break
        classification=None; err=None
        if FATAL.is_file():
            try:
                f=json.loads(FATAL.read_text(encoding='utf-8')); classification=f.get('classification'); err=f.get('error')
            except Exception: pass
        if classification=='SCIENTIFIC_FAIL_CLOSED':
            raise RuntimeError(f'SCIENTIFIC FAIL-CLOSED: {err}; see {FATAL}')
        print(f'⚠️ Runtime interruption; completed chunks are preserved. retrying... classification={classification}')
        time.sleep(8*attempt)
    if not success:
        raise RuntimeError(f'RUNTIME_RETRYABLE after 3 automatic attempts. Re-run this same notebook; valid chunks will resume. See {FATAL}')

    q=json.loads(QUALITY.read_text(encoding='utf-8'))
    if q.get('status')!='PASS' or q.get('completed_chunks')!=16 or q.get('input_rows')!=4000 or q.get('unique_races')!=2000:
        raise RuntimeError('FAIL-CLOSED Stage3 v2 final quality cardinality')
    if q.get('profile_selection_performed') is not False or q.get('market_specific_threshold_tuning_performed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage3 prereg drift')
    if q.get('result_access') is not False or q.get('payout_access') is not False or q.get('settlement_access') is not False or q.get('realized_roi_computed') is not False:
        raise RuntimeError('FAIL-CLOSED Stage3 outcome firewall')
    receipt={
      'record':'STAGE3_TICKET_FILTER_RECEIPT_v2','status':'PASS',
      'runtime_mode':'RESUMABLE_ATOMIC_CHUNKS','stage3_engine_git_blob':'91922c3f2f3da4f2af00e8e6ffcba2c6eaf041df',
      'stage3_prereg_git_blob':'ba4175bb044bcacfa66a7b8d089e92c04762b2e6',
      'runtime_reliability_standard_git_blob':'73db0eff65c9fa8ee4e97420ff3069481d359d2e',
      'stage2_catalog_sha256':EXPECTED_S2_SHA,'summary_csv_sha256':sha256(SUMMARY),'quality_sha256':sha256(QUALITY),
      'input_rows':4000,'unique_races':2000,'completed_chunks':16,'chunk_size':250,
      'profile_selection_performed':False,'result_access':False,'payout_access':False,'settlement_access':False,
      'realized_roi_computed':False,'scientific_trial_count':0,'ECON_HOLDOUT1000':'SEALED'
    }
    RECEIPT.write_text(json.dumps(receipt,ensure_ascii=False,sort_keys=True,indent=2)+'\n',encoding='utf-8')

print('✅ STAGE3 v2 PASS — RESUMABLE PIPELINE COMPLETE')
print('Drive folder:',OUT)
print('RESULT/PAYOUT/Settlement/realized ROI access = none')
print('Profile promotion = none')
print('ECON_HOLDOUT1000 = SEALED')
